# Drug dosing (PK/PD)

**What you will learn:** model a reach-avoid problem in 3D state space and compare
an HJ-synthesized dosing schedule with naive alternatives.

**pyspect API:** `TVHJImpl`, `reach(target, constraint)`

**Prerequisites:** [`reach_avoid.ipynb`](reach_avoid.ipynb)

Bring therapeutic effect to its target band without exceeding toxic drug
concentrations in blood or tissue.


In [ ]:
import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt

from scipy.integrate import solve_ivp

from pyspect.impls.hj_reachability import TVHJImpl
from pyspect.systems.hj_reachability import PKPD

In [ ]:
# Same grid as hjr_examples: 26^3 cells, 8-day horizon sampled 24 times per day
POINTS_PER_DAY = 24
T = 8
N = POINTS_PER_DAY * T

AXES = [
    dict(name='t',  bounds=[0, T], points=N + 1),
    dict(name='x1', bounds=[0, 1], points=26),
    dict(name='x2', bounds=[0, 1], points=26),
    dict(name='x3', bounds=[0, 5], points=26),
]

impl = TVHJImpl(dict(cls=PKPD, gamma=0.0, delta=0.3), AXES, accuracy='very_high')

S = impl.grid.states
X1, X2, X3 = S[..., 0], S[..., 1], S[..., 2]

# Target: therapeutic effect, |x1 - 0.53| <= 0.1
l = 10 * jnp.abs(X1 - 0.53) - 1

# Constraint: stay below the toxic threshold in blood and in tissue
g = jnp.maximum(X2 - 0.8, X3 - 0.8)

## Solve the reach-avoid

`reach(target, constraint)` is the native `TVHJImpl` primitive for this: at every step it
takes the union with the target and the intersection with the safe set, which is exactly
the `max(min(v, l), g)` post-processor used in the original script.


In [ ]:
V = impl.reach(l, g)

ttg = np.array(impl.timeline)[::-1]      # time-to-go attached to each index
V_np = np.array(V)
print('V', V_np.shape)
print(f'reachable at full horizon: {100 * (V_np[0] <= 0).mean():.1f} % of the grid')
print(f'value at the origin (untreated patient): {V_np[0, 0, 0, 0]:+.3f}')

In [ ]:
x1v = np.array(impl.grid.coordinate_vectors[0])
x2v = np.array(impl.grid.coordinate_vectors[1])
x3v = np.array(impl.grid.coordinate_vectors[2])

# The values span several units, so clip the colour scale to make the zero level readable
KW = dict(origin='lower', aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)

fig, axs = plt.subplots(1, 2, figsize=(13, 5))

sl = V_np[0, :, :, 0]                       # effect vs blood, empty tissue
ax = axs[0]
im_ = ax.imshow(sl.T, extent=[x1v[0], x1v[-1], x2v[0], x2v[-1]], **KW)
ax.contour(x1v, x2v, sl.T, levels=[0], colors='black', linewidths=1.8)
ax.axvspan(0.43, 0.63, color='green', alpha=0.25, label='Therapeutic band')
ax.axhline(0.8, color='red', ls='--', lw=1.5, label='Toxic threshold')
ax.plot(0, 0, 'ko', ms=9, label='Patient at day 0')
plt.colorbar(im_, ax=ax, label=r'$V$ (clipped)')
ax.set_xlabel(r'$x_1$  (effect)'); ax.set_ylabel(r'$x_2$  (blood)')
ax.set_title(r'Slice $x_3 = 0$')
ax.legend(loc='lower left', framealpha=1.0, fontsize=9)

sl = V_np[0, 0, :, :]                       # blood vs tissue, no effect yet
ax = axs[1]
im_ = ax.imshow(sl.T, extent=[x2v[0], x2v[-1], x3v[0], x3v[-1]], **KW)
ax.contour(x2v, x3v, sl.T, levels=[0], colors='black', linewidths=1.8)
ax.axhline(0.8, color='red', ls='--', lw=1.5, label='Toxic threshold')
ax.axvline(0.8, color='red', ls='--', lw=1.5)
ax.plot(0, 0, 'ko', ms=9, label='Patient at day 0')
ax.set_ylim(0, 1.5)
plt.colorbar(im_, ax=ax, label=r'$V$ (clipped)')
ax.set_xlabel(r'$x_2$  (blood)'); ax.set_ylabel(r'$x_3$  (tissue)')
ax.set_title(r'Slice $x_1 = 0$')
ax.legend(loc='upper right', framealpha=1.0, fontsize=9)

fig.suptitle(f'Reach-avoid set with a budget of {ttg[0]:.0f} days')
plt.tight_layout(); plt.show()

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

fig, ax = plt.subplots(figsize=(5.2, 4.6))

def update(i):
    ax.clear()
    s_ = V_np[i, :, :, 0]
    ax.imshow(s_.T, extent=[x1v[0], x1v[-1], x2v[0], x2v[-1]], **KW)
    ax.contour(x1v, x2v, s_.T, levels=[0], colors='black', linewidth=1.4)
    ax.axvspan(0.43, 0.63, color='green', alpha=0.25)
    ax.axhline(0.8, color='red', ls='--', lw=1.0)
    ax.plot(0, 0, 'ko', ms=5)
    ax.set_title(f'Budget = {ttg[i]:.1f} days')
    ax.set_xlabel(r'$x_1$  (effect)')
    ax.set_ylabel(r'$x_2$  (blood)')

ani = FuncAnimation(fig, update, frames=len(V_np), interval=150, blit=False)
plt.close(fig)
HTML(ani.to_jshtml())


## Closed-loop dosing

The gradient of `V` gives the optimal dose at every instant. The treatment is applied for
the first 8 days, then stopped, so that we can watch the drug wash out.

This notebook uses **sample-and-hold** dosing (one dose per day-step), which matches the original presentation script. See `two_target.ipynb` for the reusable `closed_loop_rhs` pattern.


In [ ]:
dyn = impl.reach_dynamics
grads = [impl.grid.grad_values(jnp.asarray(V_np[i])) for i in range(N)]

def rhs(u):
    def f(t, x):
        xs = jnp.asarray(x)
        return np.array(dyn.open_loop_dynamics(xs, 0.0)
                        + dyn.control_jacobian(xs, 0.0) @ jnp.array([u]))
    return f

ts, ys, us = [], [], []
x0 = np.array([0.0, 0.0, 0.0])
for i in range(2 * N):
    if i < N:
        gr = impl.grid.interpolate(grads[i], state=jnp.asarray(x0))
        u = float(dyn.optimal_control(jnp.asarray(x0), 0.0, gr)[0])
    else:
        u = 0.0                       # treatment stopped after day 8

    sol = solve_ivp(rhs(u), [i / POINTS_PER_DAY, (i + 1) / POINTS_PER_DAY],
                    x0, max_step=0.01)
    x0 = sol.y[:, -1]
    ts.append(sol.t if i == 0 else sol.t[1:])
    ys.append(sol.y if i == 0 else sol.y[:, 1:])
    us.append(u)

t_sol = np.concatenate(ts)
y_sol = np.concatenate(ys, axis=1)
u_sol = np.array(us)

print(f'peak blood  : {y_sol[1].max():.3f}  (toxic above 0.8)')
print(f'peak tissue : {y_sol[2].max():.3f}  (toxic above 0.8)')
print(f'effect on day 8: {np.interp(T, t_sol, y_sol[0]):.3f}  (target 0.53)')

In [ ]:
def treatment_plot(t_sol, y_sol, u_sol, u_t, title):
    fig, axs = plt.subplots(3, 1, figsize=(7, 6), sharex=True)

    axs[0].plot(u_t, u_sol, color='black')
    axs[0].set_ylabel('Dose\n(normalized)')
    axs[0].set_ylim([-0.05, 1.05])

    axs[1].plot(t_sol, y_sol[1], color='magenta', label='Blood')
    axs[1].plot(t_sol, y_sol[2], color='purple', label='Tissue')
    axs[1].axhline(0.8, color='red', ls='--', label='Toxic')
    axs[1].set_ylabel('[Drug]\n(normalized)')
    axs[1].set_ylim([-0.05, 1.55])
    axs[1].legend(loc='center left', bbox_to_anchor=(1, 0.5))

    axs[2].plot(t_sol, y_sol[0], color='blue')
    axs[2].axhline(0.5, color='green', ls='--', label='Therapeutic')
    axs[2].set_ylabel('[X]\n(normalized)')
    axs[2].set_ylim([-0.05, 1.05])
    axs[2].set_xlabel('Day')
    axs[2].set_xlim(0, 2 * T)
    axs[2].legend(loc='center left', bbox_to_anchor=(1, 0.5))

    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

treatment_plot(t_sol, y_sol, u_sol, np.arange(2 * N) / POINTS_PER_DAY,
               'Dosing synthesized from the value function')

## Three dosing schedules that fail

The point of the example is that the schedule above is not obvious. Three natural
alternatives all miss — in different ways.


In [ ]:
def open_loop(u_of_t):
    sol = solve_ivp(lambda t, x: rhs(u_of_t(t))(t, x), [0, 2 * T],
                    np.zeros(3), max_step=0.01)
    u_t = np.arange(2 * N) / POINTS_PER_DAY
    return sol.t, sol.y, np.array([u_of_t(t) for t in u_t]), u_t

t1, y1, u1, ut1 = open_loop(lambda t: 0.635 if t < T else 0.0)
treatment_plot(t1, y1, u1, ut1, 'Constant dose 0.635 for 8 days')
print(f'peak tissue: {y1[2].max():.3f}')

In [ ]:
t2, y2, u2, ut2 = open_loop(lambda t: 1.0 if t < 2.65 else 0.0)
treatment_plot(t2, y2, u2, ut2, 'Maximum dose for 2.65 days, then nothing')
print(f'peak tissue: {y2[2].max():.3f}  |  effect at day 16: {y2[0, -1]:.3f}')

In [ ]:
# Dose only after day 5.2 — too late within the 8-day budget (original: empty.svg)
sol = solve_ivp(lambda t, x: rhs(1.0 if t > 5.2 else 0.0)(t, x), [0, T],
                np.zeros(3), max_step=0.01)
u_t = np.arange(N) / POINTS_PER_DAY
u3 = np.array([1.0 if t > 5.2 else 0.0 for t in u_t])
treatment_plot(sol.t, sol.y, u3, u_t, 'Dose only after day 5.2 (within 8-day window)')
print(f'effect at day 8: {sol.y[0, -1]:.3f}  (never reaches therapeutic band)')